setup and functions

Same folders and functions you’ve used before, now in one place: ingest (Auto Loader into bronze), merge_into_silver, and merge_into_gold. new_rows(df) keeps only the bronze rows that came from the current batch’s file, which is how silver and gold process only new data.

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"
base = "/Volumes/workspace/de_ecommerce/raw_files"
holding = f"{base}/holding"
incoming = f"{base}/incoming"
ckpt = f"{base}/_checkpoints"
meta = ["source_file", "ingested_at"]
tables = ["orders", "order_items", "order_payments", "order_reviews"]

#batch = "batch_2"   # this line decides which batch runs
batch = "batch_3"   # this line decides which batch runs

def tbl(n):    return spark.table(f"{catalog}.{schema}.{n}")
def bronze(n): return tbl(f"bronze_{n}")
def silver(n): return tbl(f"silver_{n}")
def gold(n):   return tbl(f"gold_{n}")

def new_rows(df):
    return df.filter(F.col("source_file").contains(f"/{batch}/"))

def ingest(t):
    target = f"{catalog}.{schema}.bronze_{t}"
    cols = spark.table(target).columns
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{ckpt}/{t}/schema")
        .load(f"{incoming}/{t}/")
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingested_at", F.current_timestamp())
        .drop("_rescued_data")
        .select(*cols)
        .writeStream
        .option("checkpointLocation", f"{ckpt}/{t}/checkpoint")
        .trigger(availableNow=True)
        .toTable(target)
        .awaitTermination())

def merge_into_silver(df, name, keys):
    df.createOrReplaceTempView(f"src_{name}")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    res = spark.sql(f"""
        MERGE INTO {catalog}.{schema}.silver_{name} AS t
        USING src_{name} AS s
        ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    print(f"silver_{name}: inserted {res['num_inserted_rows']}, updated {res['num_updated_rows']}, total {silver(name).count()}")

def merge_into_gold(df, name, keys):
    df.createOrReplaceTempView(f"src_{name}")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    res = spark.sql(f"""
        MERGE INTO {catalog}.{schema}.gold_{name} AS t
        USING src_{name} AS s
        ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    print(f"gold_{name}: inserted {res['num_inserted_rows']}, updated {res['num_updated_rows']}, total {gold(name).count()}")

snapshot before

This records the current row count of every table we’re about to change. Run it once, before delivering the batch.

In [0]:
names = ([f"bronze_{t}" for t in tables] + [f"silver_{t}" for t in tables]
         + ["gold_fact_order_items", "gold_fact_orders"])
before = {n: tbl(n).count() for n in names}
for n in names:
    print(n, before[n])

bronze_orders 66638
bronze_order_items 75331
bronze_order_payments 69898
bronze_order_reviews 66560
silver_orders 66638
silver_order_items 75331
silver_order_payments 69898
silver_order_reviews 66560
gold_fact_order_items 75331
gold_fact_orders 66638


deliver batch_2

This copies batch_2 from holding into incoming, as if the source system had just sent new files. Run it only once.

In [0]:
for t in tables:
    dbutils.fs.cp(f"{holding}/{t}/{batch}", f"{incoming}/{t}/{batch}", recurse=True)
display(dbutils.fs.ls(f"{incoming}/orders"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/batch_1/,batch_1/,0,1793175323751
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/batch_2/,batch_2/,0,1793175323751
dbfs:/Volumes/workspace/de_ecommerce/raw_files/incoming/orders/initial/,initial/,0,1793175323751


bronze with Auto Loader

This runs Auto Loader for each table, then checks that each bronze table grew by exactly the rows in the batch_2 file.

In [0]:
for t in tables:
    ingest(t)

for t in tables:
    new_n = spark.read.parquet(f"{holding}/{t}/{batch}").count()
    now = bronze(t).count()
    status = "OK" if now == before[f"bronze_{t}"] + new_n else "MISMATCH"
    print(f"bronze_{t}: {before[f'bronze_{t}']} + {new_n} = {now}  [{status}]")

bronze_orders: 66638 + 19979 = 86617  [OK]
bronze_order_items: 75331 + 22978 = 98309  [OK]
bronze_order_payments: 69898 + 20763 = 90661  [OK]
bronze_order_reviews: 66560 + 19889 = 86449  [OK]


silver with MERGE (new rows only)

This takes only the bronze rows that came from the batch_2 file, cleans them with the same rules as before, and merges them into silver.

In [0]:
ts_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]

# ---- orders ----
df = new_rows(bronze("orders")).drop(*meta)
for c in ts_cols:
    df = df.withColumn(c, F.col(c).cast("timestamp"))
df = (df.dropDuplicates(["order_id"])
        .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
        .withColumn("order_status", F.lower(F.trim("order_status")))
        .filter(F.col("order_delivered_customer_date").isNull() |
                (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))))
merge_into_silver(df, "orders", ["order_id"])

# ---- order_items ----
df = new_rows(bronze("order_items")).drop(*meta)
df = (df.dropDuplicates(["order_id", "order_item_id"])
        .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull() & F.col("seller_id").isNotNull())
        .filter((F.col("price") > 0) & (F.col("freight_value") >= 0))
        .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp")))
merge_into_silver(df, "order_items", ["order_id", "order_item_id"])

# ---- order_payments ----
df = new_rows(bronze("order_payments")).drop(*meta)
df = (df.dropDuplicates(["order_id", "payment_sequential"])
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("payment_value") >= 0)
        .filter(F.col("payment_type") != "not_defined"))
merge_into_silver(df, "order_payments", ["order_id", "payment_sequential"])

# ---- order_reviews ----
df = new_rows(bronze("order_reviews")).drop(*meta)
df = (df.withColumn("review_score", F.col("review_score").cast("int"))
        .withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
        .withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))
        .filter(F.col("order_id").isNotNull() & F.col("review_score").between(1, 5))
        .dropDuplicates(["review_id", "order_id"]))
merge_into_silver(df, "order_reviews", ["review_id", "order_id"])

silver_orders: inserted 0, updated 19979, total 86617
silver_order_items: inserted 0, updated 22978, total 98309
silver_order_payments: inserted 0, updated 20763, total 90661
silver_order_reviews: inserted 0, updated 19889, total 86449


gold with MERGE (new orders only)

This finds the IDs of the new orders from batch_2, keeps only their rows in silver, and builds the fact rows for just those orders. Each order is matched to the customer version valid on its purchase date. The result is merged into the two fact tables.

In [0]:
new_ids = new_rows(bronze("orders")).select("order_id").distinct()

orders = silver("orders").join(new_ids, "order_id", "left_semi")
items = silver("order_items").join(new_ids, "order_id", "left_semi")
pays = silver("order_payments").join(new_ids, "order_id", "left_semi")
revs = silver("order_reviews").join(new_ids, "order_id", "left_semi")

d_cust = gold("dim_customer").select("customer_key", "customer_id", "start_date", "end_date")
d_sell = gold("dim_seller").select("seller_key", "seller_id")
d_prod = gold("dim_product").select("product_key", "product_id")
date_key = lambda c: F.date_format(c, "yyyyMMdd").cast("int")

orders_k = (orders.alias("o")
    .join(d_cust.alias("c"),
          (F.col("o.customer_id") == F.col("c.customer_id")) &
          (F.to_date("o.order_purchase_timestamp") >= F.col("c.start_date")) &
          (F.to_date("o.order_purchase_timestamp") < F.col("c.end_date")),
          "left")
    .select("o.*", "c.customer_key"))

# ---- fact_order_items ----
fact_order_items = (items
    .join(orders_k.select("order_id", "customer_key", "order_status", "order_purchase_timestamp"), "order_id")
    .join(d_sell, "seller_id", "left")
    .join(d_prod, "product_id", "left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("item_total", F.round(F.col("price") + F.col("freight_value"), 2))
    .select("order_id", "order_item_id", "customer_key", "seller_key", "product_key",
            "purchase_date_key", "order_status", "price", "freight_value", "item_total"))
merge_into_gold(fact_order_items, "fact_order_items", ["order_id", "order_item_id"])

# ---- fact_orders ----
item_agg = items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.round(F.sum("price"), 2).alias("items_value"),
    F.round(F.sum("freight_value"), 2).alias("freight_value"))
pay_agg = pays.groupBy("order_id").agg(
    F.round(F.sum("payment_value"), 2).alias("payment_value"),
    F.max("payment_installments").alias("max_installments"))
rev_agg = revs.groupBy("order_id").agg(
    F.round(F.avg("review_score"), 2).alias("avg_review_score"))

fact_orders = (orders_k
    .join(item_agg, "order_id", "left")
    .join(pay_agg, "order_id", "left")
    .join(rev_agg, "order_id", "left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("estimated_delivery_days", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
    .withColumn("delivery_delay_days", F.datediff("order_delivered_customer_date", "order_estimated_delivery_date"))
    .withColumn("is_late", F.col("delivery_delay_days") > 0)
    .select("order_id", "customer_key", "purchase_date_key", "order_status",
            "items_count", "items_value", "freight_value", "payment_value", "max_installments",
            "avg_review_score", "delivery_days", "estimated_delivery_days",
            "delivery_delay_days", "is_late"))
merge_into_gold(fact_orders, "fact_orders", ["order_id"])

gold_fact_order_items: inserted 0, updated 22978, total 98309
gold_fact_orders: inserted 0, updated 19979, total 86617


final check

For every table, this prints the count before and after and the difference. It then checks the silver and gold tables for duplicate keys and the gold facts for missing keys.

In [0]:
for n in names:
    now = tbl(n).count()
    print(f"{n}: {before[n]} -> {now} (+{now - before[n]})")

keys = {"silver_orders": ["order_id"],
        "silver_order_items": ["order_id", "order_item_id"],
        "silver_order_payments": ["order_id", "payment_sequential"],
        "silver_order_reviews": ["review_id", "order_id"],
        "gold_fact_orders": ["order_id"],
        "gold_fact_order_items": ["order_id", "order_item_id"]}
for n, k in keys.items():
    dups = tbl(n).groupBy(*k).count().filter("count > 1").count()
    print(f"{n}: duplicate keys {dups}")

for n in ["gold_fact_orders", "gold_fact_order_items"]:
    print(n, "missing customer_key:", tbl(n).filter(F.col("customer_key").isNull()).count())

bronze_orders: 66638 -> 86617 (+19979)
bronze_order_items: 75331 -> 98309 (+22978)
bronze_order_payments: 69898 -> 90661 (+20763)
bronze_order_reviews: 66560 -> 86449 (+19889)
silver_orders: 66638 -> 86617 (+19979)
silver_order_items: 75331 -> 98309 (+22978)
silver_order_payments: 69898 -> 90661 (+20763)
silver_order_reviews: 66560 -> 86449 (+19889)
gold_fact_order_items: 75331 -> 98309 (+22978)
gold_fact_orders: 66638 -> 86617 (+19979)
silver_orders: duplicate keys 0
silver_order_items: duplicate keys 0
silver_order_payments: duplicate keys 0
silver_order_reviews: duplicate keys 0
gold_fact_orders: duplicate keys 0
gold_fact_order_items: duplicate keys 0
gold_fact_orders missing customer_key: 0
gold_fact_order_items missing customer_key: 0
